# 📦 Dataset Preparation for Trajectory-Guided Training

This notebook guides you through preparing a high-quality dataset for training.

## What You'll Do

1. Upload/download videos
2. Process videos into clips
3. Extract trajectories
4. Generate captions
5. Create training CSV
6. Validate dataset quality

In [ ]:
import sys
import os
sys.path.insert(0, "/workspace/LTX_video_training")
os.chdir("/workspace/LTX_video_training")

from pathlib import Path
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"Working directory: {os.getcwd()}")

## Step 1: Download Sample Dataset

For testing, we'll use a small sample dataset. For production, you should use 500-2000 high-quality videos.

In [ ]:
# Create raw video directory
raw_video_dir = Path("/workspace/raw_videos")
raw_video_dir.mkdir(parents=True, exist_ok=True)

print(f"Upload your videos to: {raw_video_dir}")
print("\nRecommended sources:")
print("  - Pexels: https://www.pexels.com/videos/")
print("  - Pixabay: https://pixabay.com/videos/")
print("  - Your own footage")
print("\nQuality requirements:")
print("  - Resolution: 1080p or higher")
print("  - FPS: 30+")
print("  - Duration: 3-10 seconds per clip")
print("  - Diverse motion patterns")

In [ ]:
# Check uploaded videos
video_extensions = ['.mp4', '.avi', '.mov', '.mkv', '.webm']
videos = []
for ext in video_extensions:
    videos.extend(list(raw_video_dir.glob(f"*{ext}")))

print(f"Found {len(videos)} videos:")
for i, video in enumerate(videos[:10], 1):  # Show first 10
    size_mb = video.stat().st_size / (1024 * 1024)
    print(f"  {i}. {video.name} ({size_mb:.2f} MB)")

if len(videos) > 10:
    print(f"  ... and {len(videos) - 10} more")

## Step 2: Configure Dataset Preparation

In [ ]:
from scripts.prepare_dataset import DatasetPreparator

# Configuration
config = {
    'input_dir': str(raw_video_dir),
    'output_dir': './dataset',
    'visualization_type': 'multi',  # or 'flow', 'depth'
    'min_frames': 60,  # ~2 seconds at 30fps
    'max_frames': 121,  # ~4 seconds at 30fps
    'target_fps': 30,
    'target_resolution': (704, 1216),  # H x W
    'num_workers': 4  # Parallel processing
}

print("Dataset preparation configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

## Step 3: Run Dataset Preparation

This will:
- Split long videos into clips
- Extract trajectories
- Create visualizations
- Generate captions
- Create training CSV

In [ ]:
%%time

preparator = DatasetPreparator(**config)

print("Starting dataset preparation...")
print("This may take several minutes depending on video count.")
print()

preparator.prepare_dataset()

print("\n✅ Dataset preparation complete!")

## Step 4: Validate Dataset

In [ ]:
import pandas as pd
import json

# Load metadata
metadata_path = Path("./dataset/dataset_metadata.json")
if metadata_path.exists():
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
    
    print("Dataset Statistics:")
    print("=" * 60)
    print(f"Total clips: {metadata['num_clips']}")
    print(f"Failed videos: {len(metadata['failed_videos'])}")
    
    # Load training CSV
    train_csv = pd.read_csv("./dataset/train.csv")
    print(f"\nTraining samples: {len(train_csv)}")
    print("\nFirst 5 samples:")
    print(train_csv.head())
    
    # Check file sizes
    total_size = 0
    for _, row in train_csv.iterrows():
        total_size += Path(row['video_path']).stat().st_size
        total_size += Path(row['trajectory_path']).stat().st_size
    
    print(f"\nTotal dataset size: {total_size / (1024**3):.2f} GB")
else:
    print("⚠️  Metadata not found. Dataset preparation may have failed.")

## Step 5: Preview Samples

In [ ]:
from IPython.display import Video, display, HTML
import random

if 'train_csv' in locals():
    # Show random sample
    sample = train_csv.sample(n=1).iloc[0]
    
    print("Random Sample Preview:")
    print("=" * 60)
    
    # Load caption
    with open(sample['caption_path'], 'r') as f:
        caption = f.read()
    
    print(f"Caption: {caption}")
    print()
    
    # Display videos side by side
    display(HTML(f"""
    <div style="display: flex; gap: 20px;">
        <div>
            <h4>Original Video</h4>
            <video width="400" controls>
                <source src="{sample['video_path']}" type="video/mp4">
            </video>
        </div>
        <div>
            <h4>Trajectory Visualization</h4>
            <video width="400" controls>
                <source src="{sample['trajectory_path']}" type="video/mp4">
            </video>
        </div>
    </div>
    """))
else:
    print("⚠️  No training data available.")

## 🎉 Dataset Preparation Complete!

Your dataset is now ready for training.

### Dataset Location
- Videos: `./dataset/videos/`
- Trajectories: `./dataset/trajectories/`
- Captions: `./dataset/captions/`
- Training CSV: `./dataset/train.csv`

### Next Steps

1. Review sample videos to ensure quality
2. Optionally add more videos for better results
3. Open `03_Training.ipynb` to start training

### Recommendations

- **Minimum**: 100 clips for testing
- **Good**: 500-1000 clips for quality results
- **Excellent**: 2000+ clips for production